In [49]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
# import numpy as np
import great_tables as gt

In [50]:
df = pl.read_parquet("data/donnees_finales.parquet")

### Distribution des temps par sexe sous contrainte de quantile

In [51]:
def distrib_tps(quantile:float):
    """Renvoie un histogramme représentant la distribution des temps avec une précision
    à la minute près répartis selon le sexe et ne considérant que la part demandée selon
    le quantile donné."""

    # Calcul de la valeur quantile et conversion en minutes
    q = df["realTime"].quantile(quantile)
    q_min = q / 60

    # Calcul du nombre de coureurs ayant fini au-delà du temps max représenté
    nb_sup_q = df.filter(pl.col("realTime") > q).shape[0]

    # Conversion en minutes
    df_plot = df.with_columns(
        (pl.col("realTime") / 60).alias("realTime_min")
    ).to_pandas()

    # Code de l'histogramme
    sns.histplot(
        data=df_plot,
        x="realTime_min",
        hue="sex",  # couleur définie par la colonne sex
        binwidth=1,  # 1 barre = 1 minute
    )


    def format_minutes(x, pos):
        """Convertit un nmobre de secondes au format H:MM."""
        h = int(x // 60)
        m = int(x % 60)
        return f"{h}:{m:02d}"

    # Formatage des axes
    ax = plt.gca()
    ax.set_xlim(120, q_min)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(30))
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(format_minutes))

    # Légende et affichage du graphe
    plt.xlabel("Temps réel")
    plt.ylabel("Nombre de coureurs")
    plt.title(f"Distribution des temps par sexe des {quantile*100}% premiers concurrents")
    plt.show()

    # Conversion de la valeur quantile en H:MM:SS
    h = int(q) // 3600
    m = (int(q) % 3600) // 60
    s = int(q) % 60
    print(f"On observe que {nb_sup_q} coureurs ont fini en plus de {h}:{m:02d}:{s:02d}.")

### Répartitions des coureurs selon les seuils symboliques

In [52]:
def repart_seuils():
    """Renvoie un tableau généré avec great_tables représentant la répartition des
    finishers selon les seuils signficatifs de 30 minutes"""

    # Définition des seuils en secondes
    seuils = list(range(7200, 21601, 1800))  # 2h à 6h par tranche de 30 min

    # Définition des labels associés
    labels = [
        "< 2h00",  # label factice, sera filtré
        "2h00-2h30",
        "2h30-3h00",
        "3h00-3h30",
        "3h30-4h00",
        "4h00-4h30",
        "4h30-5h00",
        "5h00-5h30",
        "5h30-6h00",
        "> 6h00"
    ]

    # Création des tranches
    def get_tranche(col: str) -> pl.Expr:
        return (
            pl.col(col)
            .cut(breaks=seuils, labels=labels)
            .alias("tranche")
        )

    # Calcul du nombre total d'athlètes global et par sexe
    total = df.shape[0]
    total_h = df.filter(pl.col("sex") == "M").shape[0]
    total_f = df.filter(pl.col("sex") == "F").shape[0]

    # Aggrégation dans une nouvelle colonne selon la tranche et le sexe
    df_tranches = (
        df.with_columns(get_tranche("realTime"))
        .group_by("tranche", "sex")
        .agg(pl.len().alias("n"))
        .with_columns([
            pl.when(pl.col("sex") == "M")
            .then(pl.col("n") / total_h * 100)
            .otherwise(pl.col("n") / total_f * 100)
            .alias("pct_sex"),
            (pl.col("n") / total * 100).alias("pct_global")
        ])
    )

    # Pivot pour passer d'un df long à un df large
    df_table = (
        df_tranches
        .pivot(on="sex", index="tranche", values=["n", "pct_sex"])
        .with_columns(
            ((pl.col("n_M") + pl.col("n_F")) / total * 100).alias("pct_total")
        )
        .drop(["n_M", "n_F"])
        .sort("tranche")
    )

    # Visualisation avec great_tables
    (
        gt.GT(df_table.to_pandas(), rowname_col="tranche")
        .tab_header(title="Répartition des finishers selon les seuils significatifs")
        .fmt_number(columns=["pct_total", "pct_sex_M", "pct_sex_F"], decimals=1)
        .cols_label(
            pct_total="Global",
            pct_sex_M="Hommes",
            pct_sex_F="Femmes",
        )
        .data_color(
            columns=["pct_total", "pct_sex_M", "pct_sex_F"],
            palette="Blues",
        )
        .fmt_percent(
            columns=["pct_total", "pct_sex_M", "pct_sex_F"],
            decimals=1,
            scale_values=False
        )
        .cols_width(
            cases={
                "tranche": "100px",
                "pct_total": "75px",
                "pct_sex_M": "75px",
                "pct_sex_F": "75px",
            }
        )
        .tab_style(
            style=gt.style.borders(sides="all", color="lightgray", weight="2px"),
            locations=gt.loc.column_labels()
        )
    )